In [ ]:
!pip install -r requirements.txt

In [2]:
import numpy as np
import pandas as pd

## Clean Data from CMU Book Summaries Dataset

In [3]:
# !wget https://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz
# !tar -xvzf booksummaries.tar.gz

--2026-03-27 19:32:28--  https://www.cs.cmu.edu/~dbamman/data/booksummaries.tar.gz
Resolving www.cs.cmu.edu (www.cs.cmu.edu)... 128.2.42.95
Connecting to www.cs.cmu.edu (www.cs.cmu.edu)|128.2.42.95|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16795330 (16M) [application/x-gzip]
Saving to: ‘booksummaries.tar.gz.1’

booksummaries.tar.g 100%[===================>]  16.02M  8.46MB/s    in 1.9s    

2026-03-27 19:32:30 (8.46 MB/s) - ‘booksummaries.tar.gz.1’ saved [16795330/16795330]

booksummaries/
booksummaries/README
booksummaries/booksummaries.txt


In [5]:
cols = ['wiki_id', 'freebase_id', 'title', 'author', 'pub_date', 'genres', 'summary']
df = pd.read_csv('booksummaries/booksummaries.txt', sep='\t', header=None, names=cols)
print(df.head())

df.info()
# Drop books with no summary
df = df[df['summary'].notna() & (df['summary'].str.split().str.len() > 50)]
df.info()

   wiki_id freebase_id                                      title  \
0      620     /m/0hhy                                Animal Farm   
1      843     /m/0k36                         A Clockwork Orange   
2      986     /m/0ldx                                 The Plague   
3     1756     /m/0sww  An Enquiry Concerning Human Understanding   
4     2080     /m/0wkt                       A Fire Upon the Deep   

            author    pub_date  \
0    George Orwell  1945-08-17   
1  Anthony Burgess        1962   
2     Albert Camus        1947   
3       David Hume         NaN   
4     Vernor Vinge         NaN   

                                              genres  \
0  {"/m/016lj8": "Roman \u00e0 clef", "/m/06nbt":...   
1  {"/m/06n90": "Science Fiction", "/m/0l67h": "N...   
2  {"/m/02m4t": "Existentialism", "/m/02xlf": "Fi...   
3                                                NaN   
4  {"/m/03lrw": "Hard science fiction", "/m/06n90...   

                                           

In [12]:
# Parse genres (stored as JSON string in the raw file)
import json
def parse_genres(g):
    try:
        return list(json.loads(g).values())
    except:
        return []

df['genres_list'] = df['genres'].apply(parse_genres)
print(df['genres_list'])
df = df.reset_index(drop=True)
print(f"Books after cleaning: {len(df)}")

0        [Roman à clef, Satire, Children's literature, ...
1        [Science Fiction, Novella, Speculative fiction...
2        [Existentialism, Fiction, Absurdist fiction, N...
3                                                       []
4        [Hard science fiction, Science Fiction, Specul...
                               ...                        
15465                                                   []
15466                                                   []
15467                                  [Thriller, Fiction]
15468                                      [Autobiography]
15469              [Epistolary novel, Speculative fiction]
Name: genres_list, Length: 15470, dtype: object
Books after cleaning: 15470


In [13]:
import nltk, re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download(['stopwords','wordnet','punkt'])
stop = set(stopwords.words('english'))
lem = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [lem.lemmatize(t) for t in tokens if t not in stop]
    return ' '.join(tokens)

df['clean_summary'] = df['summary'].apply(preprocess)

df['combined'] = (df['clean_summary'] + ' ' +
                  df['author'].fillna('') + ' ' +
                  df['genres_list'].apply(lambda g: ' '.join(g)))

print(df.head())
df.to_csv('booksummaries_processed.csv', index=False)

[nltk_data] Downloading package stopwords to /home/irisx2/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/irisx2/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/irisx2/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


   wiki_id freebase_id                                      title  \
0      620     /m/0hhy                                Animal Farm   
1      843     /m/0k36                         A Clockwork Orange   
2      986     /m/0ldx                                 The Plague   
3     1756     /m/0sww  An Enquiry Concerning Human Understanding   
4     2080     /m/0wkt                       A Fire Upon the Deep   

            author    pub_date  \
0    George Orwell  1945-08-17   
1  Anthony Burgess        1962   
2     Albert Camus        1947   
3       David Hume         NaN   
4     Vernor Vinge         NaN   

                                              genres  \
0  {"/m/016lj8": "Roman \u00e0 clef", "/m/06nbt":...   
1  {"/m/06n90": "Science Fiction", "/m/0l67h": "N...   
2  {"/m/02m4t": "Existentialism", "/m/02xlf": "Fi...   
3                                                NaN   
4  {"/m/03lrw": "Hard science fiction", "/m/06n90...   

                                           

## Load data

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import joblib

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# df = pd.read_csv('booksummaries_processed.csv')
# print(df.head())

In [16]:
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['combined'])

corpus = [text.split() for text in df['clean_summary']]
bm25 = BM25Okapi(corpus)

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['clean_summary'].tolist(),
                          batch_size=64, show_progress_bar=True)

joblib.dump(tfidf_matrix, 'tfidf_matrix.pkl')

/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1167.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 242/242 [46:52<00:00, 11.62s/it]


['tfidf_matrix.pkl']

In [29]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genres_list'])

def recommend(title, matrix, df, top_k=10, weight_genre=True):
    matches = df[df['title'].str.lower() == title.lower()]
    if matches.empty:
        raise ValueError(f"Book '{title}' not found in dataset.")
    pos = matches.index[0]

    scores = cosine_similarity(matrix[pos], matrix).flatten()
    scores[pos] = 0  # exclude self

    if weight_genre:
        input_vec = genre_matrix[pos]                  # shape (n_genres,)
        overlaps = genre_matrix.dot(input_vec)         # shape (n_books,)
        scores += 0.1 * overlaps
        scores[pos] = 0                                # re-zero self after boost

    top_idx = scores.argsort()[::-1][:top_k]
    results = df.iloc[top_idx][['title', 'author', 'genres_list']].copy()
    results['score'] = scores[top_idx]
    return results.reset_index(drop=True)

In [30]:
recommend("The Fall", tfidf_matrix, df)

,title,author,genres_list,score
0,The Myth of Sisyphus,Albert Camus,[Existentialism],0.249316
1,A Happy Death,Albert Camus,[Existentialism],0.165876
2,The Stranger,Albert Camus,"[Crime Fiction, Existentialism, Fiction, Absur...",0.162963
3,The First Man,Albert Camus,[Autobiographical novel],0.152216
4,The Plague,Albert Camus,"[Existentialism, Fiction, Absurdist fiction, N...",0.151790
5,The Outsider,Colin Wilson,"[Existentialism, Philosophy]",0.136384
6,Steppenwolf,Hermann Hesse,"[Existentialism, Speculative fiction, Fiction,...",0.132286
7,"They Shoot Horses, Don't They?",Horace McCoy,[Existentialism],0.122435
8,As I Walked Out One Midsummer Morning,Laurie Lee,[],0.072403
9,The Family of Pascual Duarte,Camilo José Cela,[Novel],0.071896


In [31]:
recommend("The Fall", tfidf_matrix, df, weight_genre=False)

,title,author,genres_list,score
0,The First Man,Albert Camus,[Autobiographical novel],0.152216
1,The Myth of Sisyphus,Albert Camus,[Existentialism],0.149316
2,As I Walked Out One Midsummer Morning,Laurie Lee,[],0.072403
3,The Family of Pascual Duarte,Camilo José Cela,[Novel],0.071896
4,Notes from Underground,Fedor Mikhaïlovitch Dostoïevski,[Novella],0.071288
5,Chuang Tse and the first emperor,Anna Russo,[Spirituality],0.067823
6,Veracity,NaN,"[Suspense, Dystopia]",0.066981
7,From an Abandoned Work,Samuel Beckett,[],0.066018
8,A Happy Death,Albert Camus,[Existentialism],0.065876
9,White Nights,Fyodor Dostoyevsky,[],0.065141


## OpenLibrary Data

In [ ]:
!wget https://openlibrary.org/data/ol_dump_works_latest.txt.gz
!tar -xvzf ol_dump_works_latest.tar.gz

--2026-03-31 00:45:00--  https://openlibrary.org/data/ol_dump_works_latest.txt.gz
Resolving openlibrary.org (openlibrary.org)... 207.241.234.205
Connecting to openlibrary.org (openlibrary.org)|207.241.234.205|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://archive.org/download/ol_dump_2026-02-28/ol_dump_works_2026-02-28.txt.gz [following]
--2026-03-31 00:45:01--  https://archive.org/download/ol_dump_2026-02-28/ol_dump_works_2026-02-28.txt.gz
Resolving archive.org (archive.org)... 207.241.224.2
Connecting to archive.org (archive.org)|207.241.224.2|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://ia802803.us.archive.org/21/items/ol_dump_2026-02-28/ol_dump_works_2026-02-28.txt.gz [following]
--2026-03-31 00:45:01--  https://ia802803.us.archive.org/21/items/ol_dump_2026-02-28/ol_dump_works_2026-02-28.txt.gz
Resolving ia802803.us.archive.org (ia802803.us.archive.org)... 207.241.232.113
Connecting to ia802803.us.arc